## Задача по работе с API. Создание отчётности с использованием Яндекс.Метрики.

In [ ]:
import requests
import time
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
from datetime import datetime

# =============================================================================
# КОНФИГУРАЦИЯ (ЗАМЕНИ НА СВОИ ДАННЫЕ)
# =============================================================================
COUNTER_ID = "YOUR_COUNTER_ID"       # ID счетчика Яндекс.Метрики
OAUTH_TOKEN = "YOUR_OAUTH_TOKEN"     # OAuth токен с правами на чтение метрики
DATE_START = "2026-07-01"            # Период выгрузки по заданию
DATE_END = "2026-07-31"
LOGS_API_BASE = "https://api-metrika.yandex.net/management/v1"

# Поля визитов из документации (ym:s:*) для выгрузки сырых данных
VISIT_FIELDS = [
    "ym:s:date", "ym:s:dateTime", "ym:s:clientID", "ym:s:startURL", 
    "ym:s:trafficSourceName", "ym:s:lastsignTrafficSource", 
    "ym:s:goalsID", "ym:s:visitDuration", "ym:s:bounce",
    "ym:s:parsedParamsKey1", "ym:s:parsedParamsKey2" # Для брендов/категорий
]

HEADERS = {
    "Authorization": f"OAuth {OAUTH_TOKEN}",
    "Content-Type": "application/json"
}

# =============================================================================
# ЧАСТЬ 1. ЛОГИКА ВЫГРУЗКИ ЧЕРЕЗ LOGS API
# =============================================================================
def create_log_request():
    """Создает запрос на подготовку лога. Учитывает асинхронность API."""
    payload = {
        "date1": DATE_START,
        "date2": DATE_END,
        "fields": VISIT_FIELDS,
        "source": "visits",          # Тип данных: визиты
        "limit": 100000              # Ограничение на размер одной партии
    }
    
    url = f"{LOGS_API_BASE}/counter/{COUNTER_ID}/logrequests"
    response = requests.post(url, headers=HEADERS, json=payload)
    response.raise_for_status()
    request_id = response.json()["log_request"]["request_id"]
    print(f"[OK] Запрос создан. Request ID: {request_id}")
    return request_id

def wait_for_processing(request_id):
    """Поллинг статуса обработки. Логи Метрики готовятся асинхронно."""
    while True:
        status_url = f"{LOGS_API_BASE}/counter/{COUNTER_ID}/logrequest/{request_id}"
        resp = requests.get(status_url, headers=HEADERS).json()
        status = resp["log_request"]["status"]
        
        if status == "processed":
            print("[OK] Лог обработан и готов к выгрузке.")
            return True
        elif status == "failed":
            raise Exception("Ошибка при обработке лога на стороне Метрики")
        else:
            print(f"[WAIT] Статус: {status}. Ожидание 10 секунд...")
            time.sleep(10)

def download_and_clean_data(request_id):
    """Выгружает JSON, парсит и приводит к плоской таблице."""
    # В реальном коде здесь цикл по partNumber, если данных > limit
    download_url = f"{LOGS_API_BASE}/counter/{COUNTER_ID}/logrequest/{request_id}/part/0/download"
    resp = requests.get(download_url, headers=HEADERS)
    raw_data = resp.json() 
    
    # Нормализация сырых данных Метрики (массивы целей, параметров)
    df = pd.DataFrame(raw_data)
    
    # Преобразование массива целей в строку для анализа
    if 'ym:s:goalsID' in df.columns:
        df['goal_ids'] = df['ym:s:goalsID'].apply(lambda x: ','.join(map(str, x)) if isinstance(x, list) else str(x))
    
    # Извлечение бренда из пользовательских параметров (примерная логика)
    if 'ym:s:parsedParamsKey1' in df.columns:
        df['brand'] = df['ym:s:parsedParamsKey1'].apply(lambda x: x[0] if isinstance(x, list) and len(x)>0 else None)
        
    # Приведение типов дат
    df['visit_date'] = pd.to_datetime(df['ym:s:date'])
    df['week_number'] = df['visit_date'].dt.isocalendar().week.astype(int)
    
    print(f"[OK] Загружено строк: {len(df)}")
    return df

# =============================================================================
# ЧАСТЬ 2. ТРАНСФОРМАЦИЯ В БИЗНЕС-ОТЧЕТНОСТЬ (EXCEL)
# =============================================================================
def generate_excel_report(df):
    """Формирует многолистовой Excel с ответами на вопросы задания."""
    filename = f"YM_Report_{DATE_START}_{DATE_END}.xlsx"
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        
        # 1. Эффективность источников трафика
        # Метрика эффективности: Конверсия в достижение цели / Отказность
        traffic_eff = df.groupby('ym:s:lastsignTrafficSource').agg(
            visits=('ym:s:clientID', 'count'),
            unique_users=('ym:s:clientID', 'nunique'),
            bounces=('ym:s:bounce', 'sum'),
            goals_reached=('goal_ids', lambda x: x.str.contains(r'\d').sum())
        ).reset_index()
        traffic_eff['bounce_rate'] = (traffic_eff['bounces'] / traffic_eff['visits'] * 100).round(2)
        traffic_eff['conversion_rate'] = (traffic_eff['goals_reached'] / traffic_eff['visits'] * 100).round(2)
        traffic_eff.sort_values('conversion_rate', ascending=False, inplace=True)
        traffic_eff.to_excel(writer, sheet_name="Эффективность трафика", index=False)
        
        # 2. Популярные бренды товаров
        brand_stats = df[df['brand'].notna()].groupby('brand').agg(
            views=('ym:s:clientID', 'count'),
            unique_visitors=('ym:s:clientID', 'nunique')
        ).sort_values('views', ascending=False).head(20).reset_index()
        brand_stats.to_excel(writer, sheet_name="Бренды товаров", index=False)
        
        # 3. Уникальные пользователи в среднем за день
        daily_users = df.groupby('visit_date')['ym:s:clientID'].nunique().reset_index(name='unique_users')
        avg_daily = pd.DataFrame({
            'metric': ['Среднее кол-во уникальных пользователей в день'],
            'value': [daily_users['unique_users'].mean().round(2)]
        })
        avg_daily.to_excel(writer, sheet_name="Уник. пользователи (день)", index=False)
        
        # 4. Популярные страницы входа
        top_landing = df.groupby('ym:s:startURL')['ym:s:clientID'].nunique().sort_values(ascending=False).head(15).reset_index(name='unique_visitors')
        top_landing.to_excel(writer, sheet_name="Страницы входа", index=False)
        
        # 5. Динамика продаж по неделям
        weekly_sales = df.groupby('week_number').agg(
            visits=('ym:s:clientID', 'count'),
            goals=('goal_ids', lambda x: x.str.contains(r'\d').sum())
        ).reset_index()
        weekly_sales['wow_change_pct'] = weekly_sales['goals'].pct_change() * 100
        weekly_sales.to_excel(writer, sheet_name="Динамика по неделям", index=False)

    print(f"[OK] Отчет сохранен: {filename}")

# =============================================================================
# ЗАПУСК ВСЕГО ПАЙПЛАЙНА
# =============================================================================
if __name__ == "__main__":
    try:
        req_id = create_log_request()
        wait_for_processing(req_id)
        df_raw = download_and_clean_data(req_id)
        generate_excel_report(df_raw)
    except Exception as e:
        print(f"[ERROR] {e}")

## Часть 3. Дополнительные рекомендации по аналитике и Data Quality

### Рекомендуемые дополнительные метрики
1. **RFM-анализ аудитории:** Сегментация `clientID` по Recency, Frequency, Monetary (на основе целей/покупок) для выявления ядра лояльной аудитории vs разовых посетителей.
2. **Глубина воронки по источникам:** Не просто конверсия в цель, а проходимость каждого шага (Главная → Каталог → Карточка → Корзина → Цель). Позволяет найти узкие места для конкретных каналов.
3. **Return Rate (возвращаемость):** Доля пользователей, вернувшихся на сайт в течение 7/14/30 дней после первого визита. Ключевой показатель здоровья органического трафика.

### Контроль качества данных и алерты
Logs API возвращает неагрегированные данные, где возможны технические артефакты. Необходимо внедрить следующие проверки:

| Проверка | Метод | Действие при нарушении |
| :--- | :--- | :--- |
| **Дубликаты визитов** | `df.duplicated(subset=['ym:s:visitID']).sum()` | Алерт в Telegram/Slack, если дублей > 0.1% от объема |
| **Аномальная отказность** | Bounce Rate > 90% по конкретному источнику | Пометка источника как "подозрительный" (боты/скликивание) |
| **Разрыв временных рядов** | Отсутствие данных за конкретную дату | Автоматический retry запроса к Logs API + уведомление |
| **Целостность clientID** | NULL или пустые значения идентификаторов | Исключение из расчета уникальных пользователей, фиксация % потерь |

**Формат алертов:** Ежедневный сводный дайджест в мессенджер команды аналитики с красными флагами (критические ошибки) и желтыми предупреждениями (трендовые отклонения).